# CFIS — recuperação otimizada e pré-validação do benchmark externo

Notebook independente, extraído do histórico `Untitled49.ipynb`. Não lê dados de teste e **não executa avaliação externa**. Preserva o CFIS e V2/V4; a V3 multiclasse não foi encontrada no código-fonte histórico e usa fallback explícito ao sklearn, sujeito a auditoria. Os K congelados não são modificados.

In [ ]:
import numpy as np, pandas as pd, warnings, time, json, hashlib
from pathlib import Path
from types import SimpleNamespace
from scipy import stats
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.inspection import permutation_importance
from sklearn.datasets import load_breast_cancer, load_iris, load_diabetes
from sklearn.utils import check_random_state
from sklearn.metrics import r2_score
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/CFIS_Artigo_Diogo')
RUN = ROOT/'04_Efficiency/CFIS_Final_Benchmark/03_Final_Execution/decision_tree_final_v1'
assert (RUN/'Manifests/protocol.json').is_file(), 'Protocolo histórico não encontrado'
print('Ambiente e protocolo localizados.')

In [ ]:
# Backup do CFIS recuperado da saída da célula 2 do Untitled49.ipynb
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.inspection import permutation_importance

def CFIS_feature_selection(
    X,
    y,
    task="classification",
    model=None,
    n_repeats=10,
    random_state=42
):
    """
    Replicação do CFIS fornecido.

    Componentes:
      1. Permutation Importance absoluta
      2. Coeficientes absolutos
      3. ANOVA F (classificação) ou
         Pearson absoluto (regressão)

    Combinação:
        CFIS = (P + C + R) / 3

    Correções operacionais:
      - Modelo ajustado antes da permutação
      - Coeficientes multiclasse agregados
        pela média dos valores absolutos

    IMPORTANTE:
      X deve conter apenas dados de treinamento.
    """

    if task not in ("classification", "regression"):
        raise ValueError("Tarefa inválida.")

    # --------------------------------------------------------
    # 1. VALIDAR ENTRADAS
    # --------------------------------------------------------

    if isinstance(X, pd.DataFrame):
        feature_names = X.columns.tolist()
        X_values = X.to_numpy(dtype=float)
    else:
        X_values = np.asarray(X, dtype=float)
        feature_names = [
            f"Feature_{i}"
            for i in range(X_values.shape[1])
        ]

    y_values = np.asarray(y).ravel()

    if X_values.ndim != 2:
        raise ValueError("X deve ser bidimensional.")

    if len(X_values) != len(y_values):
        raise ValueError("X e y possuem tamanhos diferentes.")

    if not np.isfinite(X_values).all():
        raise ValueError("X contém valores não finitos.")

    if pd.isna(y_values).any():
        raise ValueError("y contém valores ausentes.")

    n_features = X_values.shape[1]

    # --------------------------------------------------------
    # 2. PADRONIZAÇÃO — APENAS TREINAMENTO
    # --------------------------------------------------------

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_values)

    # --------------------------------------------------------
    # 3. MODELO BASE
    # --------------------------------------------------------

    if model is None:
        if task == "classification":
            estimator = LogisticRegression(
                random_state=random_state,
                max_iter=1000
            )
        else:
            estimator = Lasso(
                alpha=0.01,
                random_state=random_state,
                max_iter=1000
            )
    else:
        estimator = clone(model)

    # Correção: ajustar antes da permutação.
    estimator.fit(X_scaled, y_values)

    # --------------------------------------------------------
    # 4. COMPONENTE P — PERMUTATION IMPORTANCE
    # --------------------------------------------------------

    scoring = (
        "accuracy"
        if task == "classification"
        else "r2"
    )

    perm_result = permutation_importance(
        estimator,
        X_scaled,
        y_values,
        n_repeats=n_repeats,
        random_state=random_state,
        scoring=scoring,
        n_jobs=1
    )

    permutation_scores = np.abs(
        perm_result.importances_mean
    )

    permutation_scores = np.nan_to_num(
        permutation_scores,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    # --------------------------------------------------------
    # 5. COMPONENTE C — COEFICIENTES ABSOLUTOS
    # --------------------------------------------------------

    if hasattr(estimator, "coef_"):

        coef = np.asarray(estimator.coef_)

        if coef.ndim == 1:
            coefficient_scores = np.abs(coef)

        elif coef.ndim == 2:
            # Correção operacional para multiclasse:
            # um valor por característica.
            coefficient_scores = np.mean(
                np.abs(coef),
                axis=0
            )

        else:
            raise ValueError(
                "Formato inesperado de coeficientes."
            )

    elif hasattr(estimator, "feature_importances_"):

        coefficient_scores = np.abs(
            np.asarray(
                estimator.feature_importances_
            )
        )

    else:
        coefficient_scores = np.zeros(n_features)

    if len(coefficient_scores) != n_features:
        raise ValueError(
            "Dimensão dos coeficientes incompatível."
        )

    # --------------------------------------------------------
    # 6. COMPONENTE R — ANOVA OU PEARSON
    # --------------------------------------------------------

    correlation_scores = np.zeros(n_features)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        if task == "classification":

            classes = np.unique(y_values)

            for j in range(n_features):

                groups = [
                    X_scaled[y_values == cls, j]
                    for cls in classes
                ]

                if (
                    all(len(g) > 0 for g in groups)
                    and np.var(X_scaled[:, j]) > 0
                ):
                    f_value, _ = stats.f_oneway(*groups)

                    if np.isfinite(f_value):
                        correlation_scores[j] = max(
                            0.0,
                            float(f_value)
                        )

        else:

            for j in range(n_features):

                if np.var(X_scaled[:, j]) > 0:

                    corr = np.corrcoef(
                        X_scaled[:, j],
                        y_values
                    )[0, 1]

                    if np.isfinite(corr):
                        correlation_scores[j] = abs(
                            float(corr)
                        )

    # --------------------------------------------------------
    # 7. NORMALIZAÇÃO ORIGINAL POR SOMA
    # --------------------------------------------------------

    def normalize_component(values):

        values = np.nan_to_num(
            np.asarray(values, dtype=float),
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        total = np.sum(values)

        if total > 0:
            return values / total

        return np.zeros_like(values)

    normalized_perm = normalize_component(
        permutation_scores
    )

    normalized_coef = normalize_component(
        coefficient_scores
    )

    normalized_corr = normalize_component(
        correlation_scores
    )

    # --------------------------------------------------------
    # 8. COMBINAÇÃO MATEMÁTICA ORIGINAL
    # --------------------------------------------------------

    combined_score = (
        normalized_perm
        + normalized_coef
        + normalized_corr
    ) / 3.0

    cfis_percentage = normalize_component(
        combined_score
    ) * 100.0

    # --------------------------------------------------------
    # 9. RESULTADOS
    # --------------------------------------------------------

    results = pd.DataFrame({
        "Feature": feature_names,
        "Permutation_Importance": normalized_perm,
        "Coefficient_Abs": normalized_coef,
        "Correlation_Abs": normalized_corr,
        "CFIS_Score": combined_score,
        "CFIS_Percentage": cfis_percentage
    })

    return results.sort_values(
        "CFIS_Score",
        ascending=False,
        kind="stable"
    ).reset_index(drop=True)



cfis_function = CFIS_feature_selection
cfis_globals = cfis_function.__globals__
original_permutation = cfis_globals["permutation_importance"]


In [ ]:
# Extraído sem alterações da célula histórica 9
def fast_binary_permutation_importance_v2(
    estimator, X, y, n_repeats=10, random_state=42
):
    X = np.asarray(X)
    y = np.asarray(y)

    n_samples, n_features = X.shape
    coef = estimator.coef_[0]
    classes = estimator.classes_

    base_scores = estimator.decision_function(X)
    baseline_accuracy = np.mean(estimator.predict(X) == y)

    # Mesma semente interna utilizada pelo scikit-learn.
    rng = check_random_state(random_state)
    internal_seed = rng.randint(np.iinfo(np.int32).max)

    importances = np.empty(
        (n_features, n_repeats), dtype=float
    )

    for j in range(n_features):

        feature_rng = check_random_state(internal_seed)
        shuffle_idx = np.arange(n_samples)

        # A coluna é modificada cumulativamente.
        current_column = X[:, j].copy()
        original_column = X[:, j]

        for r in range(n_repeats):

            feature_rng.shuffle(shuffle_idx)

            # CORREÇÃO: permutar a coluna já modificada.
            current_column = current_column[shuffle_idx].copy()

            updated_scores = (
                base_scores
                + (current_column - original_column) * coef[j]
            )

            predictions = np.where(
                updated_scores > 0,
                classes[1],
                classes[0]
            )

            accuracy = np.mean(predictions == y)

            importances[j, r] = (
                baseline_accuracy - accuracy
            )

    return importances

In [ ]:
# Extraído sem alterações da célula histórica 16
def fast_regression_permutation_v4(
    estimator, X, y,
    n_repeats=10,
    random_state=42
):
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    n_samples, n_features = X.shape

    coef = np.asarray(estimator.coef_).ravel()

    if coef.shape != (n_features,):
        raise ValueError("Coeficientes incompatíveis.")

    baseline_predictions = estimator.predict(X)
    baseline_r2 = r2_score(y, baseline_predictions)

    internal_seed = check_random_state(
        random_state
    ).randint(np.iinfo(np.int32).max)

    importances = np.empty(
        (n_features, n_repeats),
        dtype=float
    )

    for j in range(n_features):
        rng = check_random_state(internal_seed)
        indices = np.arange(n_samples)

        original_column = X[:, j]
        current_column = original_column.copy()

        for r in range(n_repeats):
            rng.shuffle(indices)

            # Mesma permutação cumulativa do sklearn.
            current_column = current_column[
                indices
            ].copy()

            predictions = (
                baseline_predictions
                + (current_column - original_column)
                * coef[j]
            )

            permuted_r2 = r2_score(y, predictions)

            importances[j, r] = (
                baseline_r2 - permuted_r2
            )

    return importances

In [ ]:
# V3 multiclasse não consta definida no notebook fornecido.
# Fallback fiel ao sklearn, não apresentado como V3 otimizada.
def fast_multiclass_permutation_importance_v3(estimator, X, y, n_repeats=10, random_state=42):
    return original_permutation(estimator, X, y, n_repeats=n_repeats,
        random_state=random_state, scoring='accuracy', n_jobs=1).importances


In [ ]:
# Adaptador consolidado extraído da célula histórica 19
def consolidated_permutation_adapter(
    estimator, X, y,
    n_repeats=10,
    random_state=42,
    scoring=None,
    n_jobs=None,
    **kwargs
):
    if kwargs or n_jobs not in (None, 1):
        raise ValueError("Parâmetros fora do escopo validado.")

    X_array = np.asarray(X)

    if scoring == "accuracy":
        if not hasattr(estimator, "classes_"):
            raise ValueError("Estimador sem classes_.")

        n_classes = len(estimator.classes_)

        if n_classes == 2:
            values = fast_binary_permutation_importance_v2(
                estimator, X_array, y,
                n_repeats=n_repeats,
                random_state=random_state
            )
        elif n_classes > 2:
            values = fast_multiclass_permutation_importance_v3(
                estimator, X_array, y,
                n_repeats=n_repeats,
                random_state=random_state
            )
        else:
            raise ValueError("Número de classes inválido.")

    elif scoring == "r2":
        values = fast_regression_permutation_v4(
            estimator, X_array, y,
            n_repeats=n_repeats,
            random_state=random_state
        )

    else:
        raise ValueError(f"Scoring não suportado: {scoring}")

    return SimpleNamespace(
        importances=values,
        importances_mean=values.mean(axis=1),
        importances_std=values.std(axis=1)
    )

In [ ]:
# Validação independente em 3 tarefas; nenhuma flag é forçada.
# A comparação usa exatamente os componentes do CFIS, não só a ordenação.
from sklearn.datasets import make_classification, make_regression
cases = [
 ('binaria', *make_classification(n_samples=90,n_features=8,n_informative=5,n_redundant=0,random_state=42), 'classification'),
 ('multiclasse', *make_classification(n_samples=120,n_features=9,n_informative=6,n_redundant=0,n_classes=3,n_clusters_per_class=1,random_state=42), 'classification'),
 ('regressao', *make_regression(n_samples=110,n_features=8,noise=1,random_state=42), 'regression')
]
columns=['Permutation_Importance','Coefficient_Abs','Correlation_Abs','CFIS_Score','CFIS_Percentage']
results=[]
exact_cfis=False
equivalent_1e12=False
for name,X,y,task in cases:
    try:
        cfis_globals['permutation_importance']=original_permutation
        ref=cfis_function(pd.DataFrame(X),pd.Series(y),task=task,n_repeats=10,random_state=42)
        cfis_globals['permutation_importance']=consolidated_permutation_adapter
        opt=cfis_function(pd.DataFrame(X),pd.Series(y),task=task,n_repeats=10,random_state=42)
        err=float(np.max(np.abs(ref[columns].to_numpy()-opt[columns].to_numpy())))
        rank=ref['Feature'].tolist()==opt['Feature'].tolist()
        passed=bool(err<=1e-12 and rank)
        results.append(dict(task=name,max_abs_error=err,ranking_equal=rank,passed=passed))
    finally:
        cfis_globals['permutation_importance']=original_permutation
report=pd.DataFrame(results)
display(report)
equivalent_1e12=bool(report.passed.all())
# Flag exact_cfis denota equivalência estrita nesta bateria; não equivalência histórica universal.
exact_cfis=bool(equivalent_1e12)
print('Pré-validação sintética:', 'APROVADA' if equivalent_1e12 else 'BLOQUEADA')
print('ATENÇÃO: aprovação sintética NÃO autoriza teste externo sem auditoria histórica.')

In [ ]:
# Pré-auditoria do plano congelado, sem ler amostras de teste.
with open(RUN/'Manifests/protocol.json',encoding='utf-8') as f:
    protocol=json.load(f)
print('Modelo:',protocol.get('model'))
print('Métodos:',len(protocol.get('methods',[])))
print('Protocolo SHA256:',protocol.get('protocol_sha256'))
freeze=RUN/'Results/Auditoria_Cientifica/Fase4_Consolidacao_DecisionTree/01_k_congelado_120_combinacoes.csv'
assert freeze.is_file(), 'K congelado não encontrado'
k=pd.read_csv(freeze)
print('Linhas K congelado:',len(k))
assert len(k)==120 and protocol.get('model')=='DecisionTree'
print('PRÉ-AUDITORIA OK — TESTE EXTERNO NÃO ACESSADO')

## Condição para executar o teste externo

Este notebook **não habilita automaticamente** a execução externa. Ainda é necessário comparar rankings e componentes com checkpoints históricos do CFIS, confirmar o tratamento de V3 multiclasse e revisar as pendências do executor da Fase 5D (baseline todas as características, custos e política Random_TopK). Somente depois disso executar uma versão auditada do executor externo, com checkpoints de retomada. Não executar o benchmark interno novamente.